In [ ]:
"""YOLOv8 Wind Turbine Blade Defect Detection Training
Train on Kaggle with free P100 GPU"""

import os
import shutil
from pathlib import Path

# Fix NumPy version conflict before installing ultralytics
# ultralytics installs numpy 2.x which breaks matplotlib on Kaggle
os.system("pip install 'numpy<2' -q")
os.system("pip install ultralytics -q")

from ultralytics import YOLO
import yaml

# Configuration
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
MODEL_SIZE = 'n'  # nano for faster training

# Dataset paths (Kaggle mounts datasets to /kaggle/input/)
DATASET_PATH = "/kaggle/input/datasets/ajifoster3/yolo-annotated-wind-turbines-586x371/NordTank586x371"
OUTPUT_DIR = "/kaggle/working"

print("=" * 60)
print("YOLOv8 Wind Turbine Blade Defect Detection Training")
print("=" * 60)

# Check dataset
print(f"\nDataset path: {DATASET_PATH}")
print(f"Contents: {os.listdir(DATASET_PATH)}")

images_dir = os.path.join(DATASET_PATH, "images")
labels_dir = os.path.join(DATASET_PATH, "labels")

num_images = len(os.listdir(images_dir))
num_labels = len(os.listdir(labels_dir))
print(f"Images: {num_images}, Labels: {num_labels}")

# Create train/val split structure
print("\nCreating train/val split...")
os.makedirs(f"{OUTPUT_DIR}/dataset/train/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/dataset/train/labels", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/dataset/val/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/dataset/val/labels", exist_ok=True)

# Get labeled images only
label_files = [f.replace(".txt", "") for f in os.listdir(labels_dir) if f.endswith(".txt")]
print(f"Found {len(label_files)} labeled images")

# Shuffle and split
import random
random.seed(42)
random.shuffle(label_files)

split_idx = int(len(label_files) * 0.8)
train_files = label_files[:split_idx]
val_files = label_files[split_idx:]

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

# Copy files
for split, files in [("train", train_files), ("val", val_files)]:
    for base_name in files:
        # Find image
        for ext in [".JPG", ".jpg", ".png", ".jpeg"]:
            img_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(img_path):
                shutil.copy2(img_path, f"{OUTPUT_DIR}/dataset/{split}/images/")
                shutil.copy2(os.path.join(labels_dir, base_name + ".txt"), 
                            f"{OUTPUT_DIR}/dataset/{split}/labels/")
                break

print("Dataset prepared!")

# Create dataset.yaml
dataset_config = {
    'path': f'{OUTPUT_DIR}/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'names': {
        0: 'dirt',
        1: 'damage'
    },
    'nc': 2
}

yaml_path = f"{OUTPUT_DIR}/dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(f"Created {yaml_path}")

# Initialize model
print(f"\nLoading YOLOv8{MODEL_SIZE} model...")
model = YOLO(f'yolov8{MODEL_SIZE}.pt')

# Train
print(f"\nStarting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=0,  # GPU
    project=OUTPUT_DIR,
    name='yolov8_wind_turbine',
    exist_ok=True,
    
    # Training hyperparameters
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    
    # Early stopping
    patience=15,
    
    # Save
    save=True,
    plots=True,
)

print("\n" + "=" * 60)
print("Training Complete!")
print("=" * 60)

# Copy best model to working directory for easy download
best_model = f"{OUTPUT_DIR}/yolov8_wind_turbine/weights/best.pt"
if os.path.exists(best_model):
    shutil.copy(best_model, f"{OUTPUT_DIR}/yolov8_wind_turbine_best.pt")
    print(f"\nBest model saved to: {OUTPUT_DIR}/yolov8_wind_turbine_best.pt")
    print("Download this file and upload to S3!")

# Print final metrics
print("\nFinal training results saved to:", f"{OUTPUT_DIR}/yolov8_wind_turbine/")

# EXPORT TO EDGE TPU (For Google Coral)
print("\nExporting model to Edge TPU format...")
model.export(format="edgetpu", imgsz=IMG_SIZE, int8=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 w

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 913.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv8 Wind Turbine Blade Defect Detection Training

Dataset path: /kaggle/input/datasets/ajifoster3/yolo-annotated-wind-turbines-586x371/NordTank586x371
Contents: ['labels', 'images']
Images: 13470, Labels: 2996

Creating train/val split...
Found 2996 labeled images
Train: 2396, Val: 600
Dataset prepared!
Created /kaggle/working/dataset.yaml

Loading YOLOv8n model...

Starting training...
  Epochs: 50
  Batch size: 16
  Image size: 640
Ultralytics 8.4.171 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None,

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      2.15G      1.911      1.774      1.331         43        640: 100% ━━━━━━━━━━━━ 150/150 6.1it/s 24.6s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 5.2it/s 3.7s0.2s
                   all        600       1848      0.528      0.507        0.5      0.211

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.15G      1.875      1.665      1.307         58        640: 100% ━━━━━━━━━━━━ 150/150 6.2it/s 24.2s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 5.2it/s 3.7s0.2s
                   all        600       1848      0.531      0.456      0.473       0.21

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.15G      1.827      1.615      1.292         62        640: 100% ━━━━━━━━━━━━ 150/150 6.1it/s 24.7s0.3s
                 Class     Images

pnnxparam = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2f
inline module = ultralytics.nn.modules.block.DFL
inline module = ultralytics.nn.m

NCNN: export success ✅ 7.1s, saved as '/kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model' (11.6 MB)

Export complete (7.4s)
Results saved to /kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model imgsz=640 data=/kaggle/working/dataset.yaml  
Visualize:       https://netron.app


'/kaggle/working/yolov8_wind_turbine/weights/best_ncnn_model'